# notebook for trying a few ideas

In [1]:
import numpy as np
import wandb
import pandas as pd
import torch
import torch.nn as nn
import os


# Metrics and testing
from sklearn.metrics import roc_auc_score, accuracy_score, average_precision_score
from sklearn.model_selection import train_test_split

# My utils and setup
from nn.lstmgru_mlp import LSTMGRU_MLP
from utils.dataset import BinaryDataset
from utils.utils import Utils
from utils.loader import Loader
from utils.visualizers import Visualizer
from torch.utils.data import DataLoader
from utils.cosine import generate_cosine_pattern_graphs

# Import all embedding methods
from utils.embedding_methods.betweenness import EmbedBetweenness
from utils.embedding_methods.closeness import EmbedCloseness
from utils.embedding_methods.degree import EmbedDegree
from utils.embedding_methods.forman_ricci import EmbedForman
from utils.embedding_methods.weight import EmbedWeight

In [ ]:
df = pd.read_csv('data/input/raw/labels/networkaeternity_Label.csv')
values = df.values

In [ ]:
seed = 42  # For training and consistency
my_utils = Utils()
my_utils.set_seeds(seed)

# Parameters for graph generation
num_graphs = 250 # Number of graphs for training
max_nodes = 150  # Maximum number of nodes
avg_edges_per_node = 10  # Average number of edges per node
period_days = 30  # Set the period of the cosine cycle
start_size = 10  # Starting size for nodes
max_weight = 1  # Max weight of an edge

num_buckets = 10

# Generate graphs with cosine patterns for nodes and random edges
synthetic_graphs, labels = generate_cosine_pattern_graphs(num_graphs, max_nodes, avg_edges_per_node, period_days, start_size, max_weight)

DELETE LATER

In [ ]:
from utils.loader import Loader
from utils.visualizers import Visualizer

my_loader = Loader()
dataset = 'networkadex'
activation_name = 'Weight'
my_visualizer = Visualizer(dataset='adex', task='Binary')
data, labels = my_loader.load_data(dataset, activation_name)  # Load embeddings and labels

embedding = data[0]

my_visualizer.display_embeddings(data, data, data)

## Test Embedding Methods on Cosine Graphs

In [4]:
# Set up the embedding methods for testing
activations = [EmbedBetweenness, EmbedCloseness, EmbedDegree, EmbedForman, EmbedWeight]
activation_names = ['Betweenness', 'Closeness', 'Degree', 'Forman_Weight_1', 'Weight']
datasets = ['networkaeternity', 'networkcoindash', 'networkadex', 'networkbancor', 'CollegeMsg',  'Reddit_B']

In [5]:
num_layers = [3, 2]
dropouts = [0, 0.2]
hidden_dim_1 = [64, 128, 256]
hidden_dim_2 = [32, 64, 128, 256]
mlp_dims = [32, 64]
learning_rates = [0.0001, 0.001]
l2_regularizations = [0.00001, 0.0001, 0.001]
num_epochs = 500

top_runs = {}  # Store the best models for the best runs as a tuple of (run_id, model, valid_aucroc)

# Constants
output_dim = 1  # Binary classification
input_dim = 30  # 30-dimensional embeddings
patience = 25  # Early stopping patience

csv_file_path = 'data/output/results/BinaryTesting/data/cosine_gs.csv'

# Write the header if the file doesn't already exist
if not os.path.isfile(csv_file_path):
    pd.DataFrame(columns=['run_id', 'activation', 'seed', 'hidden_dim_1', 'hidden_dim_2', 'mlp_dim', 'learning_rate', 'dropout', 'l2_regularization', 'num_layers_LSTM', 'num_layers_GRU', 'trained_epochs', 'train_loss', 'valid_loss', 'train_aucroc', 'valid_aucroc', 'train_aucpr', 'valid_aucpr', 'train_accuracy', 'valid_accuracy']).to_csv(csv_file_path, index=False)

In [ ]:
# Check distribution of labels
percent_1s = (labels.count(1) / len(labels) ) * 100
percent_0s = (labels.count(0) / len(labels) ) * 100
print(f'The labels have {percent_1s:.2f}% 1\'s and {percent_0s:.2f}% 0\'s')
print(f'The labels have {labels.count(1)} 1\'s and {labels.count(0)} 0\'s')

In [7]:
def update_top_models(top_runs, activation, run_id, model, valid_aucroc, top_x=3):
    # If not initialized, add it
    if activation not in top_runs:
        top_runs[activation] = []
    
    top_runs[activation].append({'model': model, 'run_id': run_id, 'valid_aucroc': valid_aucroc})  # Add to the list
    top_runs[activation].sort(key=lambda x: x['valid_aucroc'], reverse=True)  # Sort the list by score (descending)
    
    # Keep only the top x models
    if len(top_runs[activation]) > top_x:
        top_runs[activation].pop()
    
    return top_runs

Regression Testing

In [ ]:
for activation, activation_name in zip(activations, activation_names):
    
    # Only testing Forman-Ricci right now
    if activation != EmbedForman:
        continue 
    
    my_activation = activation(num_buckets=num_buckets)    
    # Since Forman Ricci requires directed edges
    if activation==EmbedForman:
        embeddings = my_activation.process_graphs_for_embeddings(synthetic_graphs, is_directed=False)
    else:
        embeddings = my_activation.process_graphs_for_embeddings(synthetic_graphs)
    
    wandb.init(
        project="embedding_cosine_gs", 
        name=f"{activation_name}_buckets{num_buckets}", 
        reinit=True
    )
    
    # Split data 70/15/15
    X_train, X_tmp, y_train, y_tmp = train_test_split(embeddings, labels, test_size=0.3, shuffle=False)
    X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, shuffle=False)

    train_dataset = BinaryDataset(X_train, y_train)
    valid_dataset = BinaryDataset(X_val, y_val)
    test_dataset = BinaryDataset(X_test, y_test)
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=False, drop_last=False)  # On specific datasets the shape mismatch will crash the code on some batches
    valid_loader = DataLoader(valid_dataset, batch_size=16, shuffle=False, drop_last=False)  # drop_last fixes this issue
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, drop_last=False)
    best_valid_aucroc = float('-inf')  # Init
    
    #split_index = len(train_loader) + len(valid_loader)
    for num_layer in num_layers:
        for dropout in dropouts:
            for hidden_1 in hidden_dim_1:
                for hidden_2 in hidden_dim_2:
                    for mlp_dim in mlp_dims:
                        for lr_val in learning_rates:  
                            for l2_val in l2_regularizations:
                                curr_batch_best_aucroc = float('-inf')  # Init
                    
                                # Initialize wandb
                                run = wandb.init(project="embedding_cosine_gs", config={
                                    'activation': activation_name,
                                    'num_layers': num_layer,
                                    'dropout': dropout,
                                    'l2_regularization': l2_val,
                                    'hidden_dim_1': hidden_1,
                                    'hidden_dim_2': hidden_2,
                                    'mlp_dim': mlp_dim,
                                    'learning_rate': lr_val,
                                    'seed': seed
                                })
                                
                                
                                # Setup
                                no_improvement_counter = 0  # Number of epochs that we haven't seen an improvement in the validation AUCROC
                                model = LSTMGRU_MLP(input_dim, output_dim, hidden_dim_1=hidden_1, hidden_dim_2=hidden_2, mlp_dim=mlp_dim, dropout=dropout, num_layers_LSTM=num_layer, num_layers_GRU=num_layer)
                                optimizer = torch.optim.Adam(model.parameters(), lr=lr_val, weight_decay=l2_val)
                                criterion = nn.BCELoss()    
                                
                                for epoch in range(num_epochs):
                                    model.train()
                                    train_loss, train_aucroc, train_aucpr, train_accuracy = model.train_model_binary(model, train_loader, optimizer, criterion)

                                    with torch.no_grad():
                                        model.eval()
                                        val_preds = []
                                        valid_loss = 0
                                        for x, y in valid_loader:
                                            output = model(x)  # Maintain hidden state across time steps
                                            y = y.squeeze().float()
                                            loss = criterion(output, y)
                                            valid_loss += loss.item()
                                            val_preds.append(output.detach().numpy())

                                        # Compute metrics
                                        valid_loss /= len(valid_loader)
                                        val_preds = np.concatenate(val_preds, axis=0)  # Ensure val_preds is a flat array
                                        val_preds = np.array(val_preds)
                                        valid_aucroc = roc_auc_score(y_val, val_preds)
                                        valid_aucpr = average_precision_score(y_val, val_preds)
                                        val_pred_labels = [1 if prob >= 0.5 else 0 for prob in val_preds]  # Since accuracy requires labels
                                        valid_accuracy = accuracy_score(y_val, val_pred_labels)
                                        
                                    # Log each epoch results
                                    wandb.log({
                                        'epoch': epoch,
                                        'train_loss': train_loss,
                                        'valid_loss': valid_loss,
                                        'train_aucroc': train_aucroc,
                                        'valid_aucroc': valid_aucroc,
                                        'train_aucpr': train_aucpr,
                                        'valid_aucpr': valid_aucpr,
                                        'train_accuracy': train_accuracy,
                                        'valid_accuracy': valid_accuracy
                                    })

                                    # Optimize for the best aucroc
                                    if valid_aucroc >= curr_batch_best_aucroc:
                                        # Save for dataframe
                                        best_moment_row = {
                                            'run_id': run.name,  # For checking Wandb Logs
                                            'activation': activation_name,
                                            'seed': seed,
                                            'hidden_dim_1': hidden_1,
                                            'hidden_dim_2': hidden_2,
                                            'mlp_dim': mlp_dim,
                                            'learning_rate': lr_val,
                                            'dropout': dropout,
                                            'l2_regularization': l2_val,
                                            'num_layers_LSTM': num_layer,
                                            'num_layers_GRU': num_layer,
                                            'trained_epochs': epoch + 1,
                                            'train_loss': train_loss,
                                            'valid_loss': valid_loss,
                                            'train_aucroc': train_aucroc,
                                            'valid_aucroc': valid_aucroc,
                                            'train_aucpr': train_aucpr,
                                            'valid_aucpr': valid_aucpr,
                                            'train_accuracy': train_accuracy,
                                            'valid_accuracy': valid_accuracy
                                        }
                                        
                                        # Save the model
                                        top_runs = update_top_models(top_runs, activation, run.name, model, valid_aucroc, top_x=3)
                                        curr_batch_best_aucroc = valid_aucroc
                                        
                                        # If we have a new best model for this activation
                                        if valid_aucroc > best_valid_aucroc:
                                            best_valid_aucroc = valid_aucroc
                                            print(f'We have a new best model with a Validation AUCROC: {valid_aucroc}')
                                            test_loss, test_aucroc, test_aucpr, test_accuracy = model.test_model_binary(model, test_loader, criterion, y_test)
                                            print(f"""\tTest Loss: {valid_loss}\n\tTest Validation AUCROC: {test_aucroc}\n\tTest AUCPR: {test_aucpr}\n\tTest Accuracy: {test_accuracy}\n
                                            """)
                                    
                                    # Early stopping only after 50 epochs
                                    if epoch >= 50:
                                        if valid_aucroc >= curr_batch_best_aucroc:
                                            no_improvement_counter = 0
                                            curr_batch_best_aucroc = valid_aucroc
                                        else:
                                            no_improvement_counter += 1
                                            
                                        if no_improvement_counter == patience:
                                            print(f'Training ending at epoch number: {epoch + 1}')
                                            break
                                        
                                    # Display current results
                                    if epoch % 5 - 4 == 0:
                                        print(f"""
                                            Epoch {epoch+1}/{num_epochs}:\n\tTrain Loss: {train_loss}, Validation Loss: {valid_loss}\n\tTrain AUCROC: {train_aucroc}, Validation AUCROC: {valid_aucroc}\n\tTrain AUCPR: {train_aucpr}, Validation AUCPR: {valid_aucpr}\n\tTrain Accuracy: {train_accuracy}, Validation Accuracy: {valid_accuracy}\n
                                        """)

                                # Save the best moment from this training
                                pd.DataFrame([best_moment_row]).to_csv(csv_file_path, mode='a', header=False, index=False)
                                        
    wandb.finish()  # Close run

See summary of best runs

In [ ]:
import pandas as pd
csv_file_path = 'data/output/results/BinaryTesting/data/embedding_testing.csv'

df = pd.read_csv(csv_file_path)

tmp_df = df[df['activation'] == 'Weight']  # If you want to filter by a certain activation

pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', 1000)        # Set the maximum width for the display
pd.set_option('display.max_colwidth', None) # Show full column content if truncated

top_5_loss = df.nsmallest(6, 'valid_loss')

# Top 5 based on highest valid_aucroc (descending order)
top_5_aucroc = df.nlargest(6, 'valid_aucroc')

# Top 5 based on highest valid_accuracy (descending order)
top_5_accuracy = df.nlargest(6, 'valid_accuracy')

# Top 5 based on highest valid_aucpr (descending order)
top_5_aucpr = df.nlargest(6, 'valid_aucpr')

# Print the results
print("Top 10 rows based on lowest valid_loss:")
print(top_5_loss)

print("\nTop 10 rows based on highest valid_aucroc:")
print(top_5_aucroc)

print("\nTop 10 rows based on highest valid_accuracy:")
print(top_5_accuracy)

print("\nTop 10 rows based on highest valid_aucpr:")
print(top_5_aucpr)# Compute averages, stdev, metrics

## Test the top 3 models from each activation

In [9]:
# Save the models first
for activation, info in top_runs.items():
    for rank, entry in enumerate(info, start=1):  # To get the model ranking
        
        # Since I did the naming wrong
        if activation == EmbedBetweenness:
            activation_name = "Betweenness"
        elif activation == EmbedCloseness:
            activation_name = "Closeness"
        elif activation == EmbedDegree:
            activation_name = "Degree"
        elif activation == EmbedForman:
            activation_name = "FormanRicci"
        elif activation == EmbedWeight:
            activation_name = "Weight"
        run_name = entry['run_id'] 
        model = entry['model']  
        
        # Run name is here for clarity and reference
        torch.save(model.state_dict(), f"data/output/cached_model/BinaryTesting/{run_name}_{activation_name}_{rank}.pt")

Set up embeddings

In [10]:
my_activation = EmbedCloseness(num_buckets=num_buckets)   
closeness_embeddings = my_activation.process_graphs_for_embeddings(synthetic_graphs)
my_activation = EmbedDegree(num_buckets=num_buckets)   
degree_embeddings = my_activation.process_graphs_for_embeddings(synthetic_graphs)
my_activation = EmbedBetweenness(num_buckets=num_buckets)   
betweenness_embeddings = my_activation.process_graphs_for_embeddings(synthetic_graphs)
my_activation = EmbedForman(num_buckets=num_buckets)   
forman_embedddings = my_activation.process_graphs_for_embeddings(synthetic_graphs, is_directed=False)
my_activation = EmbedWeight(num_buckets=num_buckets)   
weight_embeddings = my_activation.process_graphs_for_embeddings(synthetic_graphs)

X_train_closeness, X_tmp_closeness, y_train_closeness, y_tmp_closeness = train_test_split(closeness_embeddings, labels, test_size=0.3, shuffle=False)
X_val_closeness, X_test_closeness, y_val_closeness, y_test_closeness = train_test_split(X_tmp_closeness, y_tmp_closeness, test_size=0.5, shuffle=False)
train_dataset = BinaryDataset(X_train_closeness, y_train_closeness)
valid_dataset = BinaryDataset(X_val_closeness, y_val_closeness)
test_dataset = BinaryDataset(X_test_closeness, y_test_closeness)
train_closeness = DataLoader(train_dataset, batch_size=16, shuffle=False, drop_last=False)  # On specific datasets the shape mismatch will crash the code on some batches
valid_closeness = DataLoader(valid_dataset, batch_size=16, shuffle=False, drop_last=False)  # drop_last fixes this issue
test_closeness = DataLoader(test_dataset, batch_size=16, shuffle=False, drop_last=False)

X_train_degree, X_tmp_degree, y_train_degree, y_tmp_degree = train_test_split(degree_embeddings, labels, test_size=0.3, shuffle=False)
X_val_degree, X_test_degree, y_val_degree, y_test_degree = train_test_split(X_tmp_degree, y_tmp_degree, test_size=0.5, shuffle=False)
train_dataset = BinaryDataset(X_train_degree, y_train_degree)
valid_dataset = BinaryDataset(X_val_degree, y_val_degree)
test_dataset = BinaryDataset(X_test_degree, y_test_degree)
train_degree = DataLoader(train_dataset, batch_size=16, shuffle=False, drop_last=False)  # On specific datasets the shape mismatch will crash the code on some batches
valid_degree = DataLoader(valid_dataset, batch_size=16, shuffle=False, drop_last=False)  # drop_last fixes this issue
test_degree = DataLoader(test_dataset, batch_size=16, shuffle=False, drop_last=False)

X_train_betweenness, X_tmp_betweenness, y_train_betweenness, y_tmp_betweenness = train_test_split(betweenness_embeddings, labels, test_size=0.3, shuffle=False)
X_val_betweenness, X_test_betweenness, y_val_betweenness, y_test_betweenness = train_test_split(X_tmp_betweenness, y_tmp_betweenness, test_size=0.5, shuffle=False)
train_dataset = BinaryDataset(X_train_betweenness, y_train_betweenness)
valid_dataset = BinaryDataset(X_val_betweenness, y_val_betweenness)
test_dataset = BinaryDataset(X_test_betweenness, y_test_betweenness)
train_betweenness = DataLoader(train_dataset, batch_size=16, shuffle=False, drop_last=False)  # On specific datasets the shape mismatch will crash the code on some batches
valid_betweenness = DataLoader(valid_dataset, batch_size=16, shuffle=False, drop_last=False)  # drop_last fixes this issue
test_betweenness = DataLoader(test_dataset, batch_size=16, shuffle=False, drop_last=False)

X_train_forman, X_tmp_forman, y_train_forman, y_tmp_forman = train_test_split(forman_embedddings, labels, test_size=0.3, shuffle=False)
X_val_forman, X_test_forman, y_val_forman, y_test_forman = train_test_split(X_tmp_forman, y_tmp_forman, test_size=0.5, shuffle=False)
train_dataset = BinaryDataset(X_train_forman, y_train_forman)
valid_dataset = BinaryDataset(X_val_forman, y_val_forman)
test_dataset = BinaryDataset(X_test_forman, y_test_forman)
train_forman = DataLoader(train_dataset, batch_size=16, shuffle=False, drop_last=False)  # On specific datasets the shape mismatch will crash the code on some batches
valid_forman = DataLoader(valid_dataset, batch_size=16, shuffle=False, drop_last=False)  # drop_last fixes this issue
test_forman = DataLoader(test_dataset, batch_size=16, shuffle=False, drop_last=False)

X_train_weight, X_tmp_weight, y_train_weight, y_tmp_weight = train_test_split(weight_embeddings, labels, test_size=0.3, shuffle=False)
X_val_weight, X_test_weight, y_val_weight, y_test_weight = train_test_split(X_tmp_closeness, y_tmp_closeness, test_size=0.5, shuffle=False)
train_dataset = BinaryDataset(X_train_weight, y_train_weight)
valid_dataset = BinaryDataset(X_val_weight, y_val_weight)
test_dataset = BinaryDataset(X_test_weight, y_test_weight)
train_weight = DataLoader(train_dataset, batch_size=16, shuffle=False, drop_last=False)  # On specific datasets the shape mismatch will crash the code on some batches
valid_weight = DataLoader(valid_dataset, batch_size=16, shuffle=False, drop_last=False)  # drop_last fixes this issue
test_weight = DataLoader(test_dataset, batch_size=16, shuffle=False, drop_last=False)

Test on Test Split

In [ ]:
# Test on these models
path = 'data/output/cached_model/BinaryTesting'
model_files = [f for f in os.listdir(path) if f.endswith(".pt") and os.path.isfile(os.path.join(path, f))]


models = {}
for model_file in model_files:
    file_path = os.path.join(path, model_file)
    model_name = os.path.splitext(model_file)[0]  # Use file name without extension as key
    models[model_name] = torch.load(file_path)

for model_name, state in models.items():
    run = model_name.split('_')[0]
    row = df[df['run_id'] == run]
    
    # Get hyperparameters:
    hidden_1 = int(row['hidden_dim_1'].values[0])
    hidden_2 = int(row['hidden_dim_2'].values[0])
    mlp_dim = int(row['mlp_dim'].values[0])
    dropout = row['dropout'].values[0]
    num_layer_LSTM = int(row['num_layers_LSTM'].values[0])
    num_layer_GRU = int(row['num_layers_GRU'].values[0])

    
    curr_model = LSTMGRU_MLP(input_dim=30, output_dim=1, hidden_dim_1=hidden_1, hidden_dim_2=hidden_2, mlp_dim=mlp_dim, dropout=dropout, num_layers_LSTM=num_layer_LSTM, num_layers_GRU=num_layer_GRU)
    curr_model.load_state_dict(state)
    curr_model.eval()
    
    # Choose proper data to load
    if 'Closeness' in model_name:
        y_test = y_test_closeness
        test_loader = test_closeness
    elif 'Degree' in model_name:
        y_test = y_test_degree
        test_loader = test_degree
    elif 'Betweenness' in model_name:
        y_test = y_test_betweenness
        test_loader = test_betweenness
    elif 'FormanRicci' in model_name:
        y_test = y_test_forman
        test_loader = test_forman
    elif 'Weight' in model_name:
        y_test = y_test_weight
        test_loader = test_weight
    
    print(f'The model: {model_name} has the following results:')
    criterion = nn.BCELoss()
    test_loss, test_aucroc, test_aucpr, test_accuracy = curr_model.test_model_binary(curr_model,test_loader=test_loader, criterion=criterion, y_test=y_test)
    print(f'\tTest Loss: {test_loss}')
    print(f'\tTest AUCROC: {test_aucroc}')
    print(f'\tTest AUCPR: {test_aucpr}')
    print(f'\tTest Accuracy: {test_accuracy}')

Try new seeds to ensure consistency

In [ ]:
runs = []

# Get all names
for model_name in models.keys():
    runs.append(model_name[:model_name.find('_')])
    
runs = np.unique(runs)

df = df[df['run_id'].isin(runs)]
print(df)

## Test model parameters, but on different seeds

In [14]:

csv_file_path = 'data/output/results/BinaryTesting/data/cosine_seed_testing.csv'

# Write the header if the file doesn't already exist
if not os.path.isfile(csv_file_path):
    pd.DataFrame(columns=['run_id', 'activation', 'seed', 'hidden_dim_1', 'hidden_dim_2', 'mlp_dim', 'learning_rate', 'dropout', 'l2_regularization', 'num_layers_LSTM', 'num_layers_GRU', 'trained_epochs', 'train_loss', 'valid_loss', 'train_aucroc', 'valid_aucroc', 'train_aucpr', 'valid_aucpr', 'train_accuracy', 'valid_accuracy', 'test_loss', 'test_aucroc', 'test_aucpr', 'test_accuracy']).to_csv(csv_file_path, index=False)

In [ ]:
testing_seeds = [42, 9999, 1, 5555, 1000]

# Test models immediately as we go
wandb.init(
    project="cosine_seed_testing", 
    reinit=True
)

for run in runs:
    row = df[df['run_id'] == str(run)]
    activation = row['activation'].values[0]
    
    if 'Closeness' == activation:
        train_loader = train_closeness
        valid_loader = valid_closeness
        test_loader = test_closeness
        y_train = y_train_closeness
        y_val = y_val_closeness
        y_test = y_test_closeness
    elif 'Degree' == activation:
        train_loader = train_degree
        valid_loader = valid_degree
        test_loader = test_degree
        y_train = y_train_degree
        y_val = y_val_degree
        y_test = y_test_degree
    elif 'Betweenness' == activation:
        train_loader = train_betweenness
        valid_loader = valid_betweenness
        test_loader = test_betweenness
        y_train = y_train_betweenness
        y_val = y_val_betweenness
        y_test = y_test_betweenness
    elif 'FormanRicci' == activation:
        train_loader = train_forman
        valid_loader = valid_forman
        test_loader = test_forman
        y_train = y_train_forman
        y_val = y_val_forman
        y_test = y_test_forman
    elif 'Weight' == activation:
        train_loader = train_weight
        valid_loader = valid_weight
        test_loader = test_weight
        y_train = y_train_weight
        y_val = y_val_weight
        y_test = y_test_weight
    
    # Get hyperparameters:
    hidden_1 = int(row['hidden_dim_1'].values[0])
    hidden_2 = int(row['hidden_dim_2'].values[0])
    mlp_dim = int(row['mlp_dim'].values[0])
    dropout = row['dropout'].values[0]
    l2_val = row['l2_regularization'].values[0]
    lr_val = row['learning_rate'].values[0]
    num_layer_LSTM = int(row['num_layers_LSTM'].values[0])
    num_layer_GRU = int(row['num_layers_GRU'].values[0])
    
        
    for seed in testing_seeds:
        np.random.seed(seed)  # Set the seed here
        
        curr_model = LSTMGRU_MLP(input_dim=30, output_dim=1, hidden_dim_1=hidden_1, hidden_dim_2=hidden_2, mlp_dim=mlp_dim, dropout=dropout, num_layers_LSTM=num_layer_LSTM, num_layers_GRU=num_layer_GRU)
        
        curr_batch_best_aucroc = float('-inf')  # Init

        # Initialize wandb
        run = wandb.init(project="cosine_seed_testing", config={
            'seed': seed,
            'activation': activation,
            'num_layers': num_layer_LSTM,
            'dropout': dropout,
            'l2_regularization': l2_val,
            'hidden_dim_1': hidden_1,
            'hidden_dim_2': hidden_2,
            'mlp_dim': mlp_dim,
            'learning_rate': lr_val,
            'seed': seed
        })

        # Setup
        no_improvement_counter = 0  # Number of epochs that we haven't seen an improvement in the validation AUCROC
        model = LSTMGRU_MLP(input_dim, output_dim, hidden_dim_1=hidden_1, hidden_dim_2=hidden_2, mlp_dim=mlp_dim, dropout=dropout, num_layers_LSTM=num_layer_LSTM, num_layers_GRU=num_layer_GRU)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr_val, weight_decay=l2_val)
        criterion = nn.BCELoss()    

        for epoch in range(num_epochs):
            model.train()
            train_loss, train_aucroc, train_aucpr, train_accuracy = model.train_model_binary(model, train_loader, optimizer, criterion)

            with torch.no_grad():
                model.eval()
                val_preds = []
                valid_loss = 0
                for x, y in valid_loader:
                    output = model(x)  # Maintain hidden state across time steps
                    y = y.squeeze().float()
                    loss = criterion(output, y)
                    valid_loss += loss.item()
                    val_preds.append(output.detach().numpy())

                # Compute metrics
                valid_loss /= len(valid_loader)
                val_preds = np.concatenate(val_preds, axis=0)  # Ensure val_preds is a flat array
                val_preds = np.array(val_preds)
                valid_aucroc = roc_auc_score(y_val, val_preds)
                valid_aucpr = average_precision_score(y_val, val_preds)
                val_pred_labels = [1 if prob >= 0.5 else 0 for prob in val_preds]  # Since accuracy requires labels
                valid_accuracy = accuracy_score(y_val, val_pred_labels)
                
            # Log each epoch results
            wandb.log({
                'epoch': epoch,
                'train_loss': train_loss,
                'valid_loss': valid_loss,
                'train_aucroc': train_aucroc,
                'valid_aucroc': valid_aucroc,
                'train_aucpr': train_aucpr,
                'valid_aucpr': valid_aucpr,
                'train_accuracy': train_accuracy,
                'valid_accuracy': valid_accuracy
            })

            # Optimize for the best aucroc
            if valid_aucroc >= curr_batch_best_aucroc:
                best_model = model
                # Save for dataframe
                best_moment_row = {
                    'run_id': run.name,  # For checking Wandb Logs
                    'activation': activation,
                    'seed': seed,
                    'hidden_dim_1': hidden_1,
                    'hidden_dim_2': hidden_2,
                    'mlp_dim': mlp_dim,
                    'learning_rate': lr_val,
                    'dropout': dropout,
                    'l2_regularization': l2_val,
                    'num_layers_LSTM': num_layer_LSTM,
                    'num_layers_GRU': num_layer_GRU,
                    'trained_epochs': epoch + 1,
                    'train_loss': train_loss,
                    'valid_loss': valid_loss,
                    'train_aucroc': train_aucroc,
                    'valid_aucroc': valid_aucroc,
                    'train_aucpr': train_aucpr,
                    'valid_aucpr': valid_aucpr,
                    'train_accuracy': train_accuracy,
                    'valid_accuracy': valid_accuracy
                }
                
                # Save the model
                curr_batch_best_aucroc = valid_aucroc
                
            
            # Early stopping only after 50 epochs
            if epoch >= 50:
                if valid_aucroc >= curr_batch_best_aucroc:
                    no_improvement_counter = 0
                    curr_batch_best_aucroc = valid_aucroc
                else:
                    no_improvement_counter += 1
                    
                if no_improvement_counter == patience:
                    print(f'Training ending at epoch number: {epoch + 1}')
                    break
                
            # Display current results
            if epoch % 5 - 4 == 0:
                print(f"""
                    Epoch {epoch+1}/{num_epochs}:\n\tTrain Loss: {train_loss}, Validation Loss: {valid_loss}\n\tTrain AUCROC: {train_aucroc}, Validation AUCROC: {valid_aucroc}\n\tTrain AUCPR: {train_aucpr}, Validation AUCPR: {valid_aucpr}\n\tTrain Accuracy: {train_accuracy}, Validation Accuracy: {valid_accuracy}\n
                """)

        test_loss, test_aucroc, test_aucpr, test_accuracy = best_model.test_model_binary(best_model, test_loader, criterion, y_test)

        best_moment_row['test_loss'] = test_loss
        best_moment_row['test_aucroc'] = test_aucroc
        best_moment_row['test_aucpr'] = test_aucpr
        best_moment_row['test_accuracy'] = test_accuracy

        # Save the best moment from this training
        pd.DataFrame([best_moment_row]).to_csv(csv_file_path, mode='a', header=False, index=False)
        
        
wandb.finish()  # Close this round of tests                            

In [16]:

csv_file_path = 'data/output/results/BinaryTesting/data/cosine_best_testing.csv'

# Write the header if the file doesn't already exist
if not os.path.isfile(csv_file_path):
    pd.DataFrame(columns=['run_id', 'activation', 'seed', 'hidden_dim_1', 'hidden_dim_2', 'mlp_dim', 'learning_rate', 'dropout', 'l2_regularization', 'num_layers_LSTM', 'num_layers_GRU', 'trained_epochs', 'train_loss', 'valid_loss', 'train_aucroc', 'valid_aucroc', 'train_aucpr', 'valid_aucpr', 'train_accuracy', 'valid_accuracy', 'test_loss', 'test_aucroc', 'test_aucpr', 'test_accuracy']).to_csv(csv_file_path, index=False)

In [ ]:
# Best models based on different criteria
df = pd.read_csv('data/output/results/BinaryTesting/data/cosine_gs.csv')

top_5_loss = df.nsmallest(25, 'valid_loss')
top_5_aucroc = df.nlargest(25, 'valid_aucroc')
top_5_accuracy = df.nlargest(25, 'valid_accuracy')
top_5_aucpr = df.nlargest(25, 'valid_aucpr')

tmp_df = pd.concat([top_5_loss, top_5_aucroc, top_5_accuracy, top_5_aucpr], axis=0)
print(tmp_df)

# Test models immediately as we go
wandb.init(
    project="cosine_best_testing", 
    reinit=True
)

In [ ]:
# Now test based on parameter results from the csv

for idx, row in tmp_df.iterrows():
    activation = row['activation']
    
    if 'Closeness' == activation:
        train_loader = train_closeness
        valid_loader = valid_closeness
        test_loader = test_closeness
        y_train = y_train_closeness
        y_val = y_val_closeness
        y_test = y_test_closeness
    elif 'Degree' == activation:
        train_loader = train_degree
        valid_loader = valid_degree
        test_loader = test_degree
        y_train = y_train_degree
        y_val = y_val_degree
        y_test = y_test_degree
    elif 'Betweenness' == activation:
        train_loader = train_betweenness
        valid_loader = valid_betweenness
        test_loader = test_betweenness
        y_train = y_train_betweenness
        y_val = y_val_betweenness
        y_test = y_test_betweenness
    elif 'FormanRicci' == activation:
        train_loader = train_forman
        valid_loader = valid_forman
        test_loader = test_forman
        y_train = y_train_forman
        y_val = y_val_forman
        y_test = y_test_forman
    elif 'Weight' == activation:
        train_loader = train_weight
        valid_loader = valid_weight
        test_loader = test_weight
        y_train = y_train_weight
        y_val = y_val_weight
        y_test = y_test_weight
    
    # Get hyperparameters:
    hidden_1 = int(row['hidden_dim_1'])
    hidden_2 = int(row['hidden_dim_2'])
    mlp_dim = int(row['mlp_dim'])
    dropout = row['dropout']
    l2_val = row['l2_regularization']
    lr_val = row['learning_rate']
    num_layer_LSTM = int(row['num_layers_LSTM'])
    num_layer_GRU = int(row['num_layers_GRU'])
    
        
    for seed in testing_seeds:        
        curr_model = LSTMGRU_MLP(input_dim=30, output_dim=1, hidden_dim_1=hidden_1, hidden_dim_2=hidden_2, mlp_dim=mlp_dim, dropout=dropout, num_layers_LSTM=num_layer_LSTM, num_layers_GRU=num_layer_GRU)
        
        curr_batch_best_aucroc = float('-inf')  # Init

        # Initialize wandb
        run = wandb.init(project="cosine_best_testing", config={
            'seed': seed,
            'activation': activation,
            'num_layers': num_layer_LSTM,
            'dropout': dropout,
            'l2_regularization': l2_val,
            'hidden_dim_1': hidden_1,
            'hidden_dim_2': hidden_2,
            'mlp_dim': mlp_dim,
            'learning_rate': lr_val,
            'seed': seed
        })

        # Setup
        no_improvement_counter = 0  # Number of epochs that we haven't seen an improvement in the validation AUCROC
        model = LSTMGRU_MLP(input_dim, output_dim, hidden_dim_1=hidden_1, hidden_dim_2=hidden_2, mlp_dim=mlp_dim, dropout=dropout, num_layers_LSTM=num_layer_LSTM, num_layers_GRU=num_layer_GRU)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr_val, weight_decay=l2_val)
        criterion = nn.BCELoss()    

        for epoch in range(num_epochs):
            model.train()
            train_loss, train_aucroc, train_aucpr, train_accuracy = model.train_model_binary(model, train_loader, optimizer, criterion)

            with torch.no_grad():
                model.eval()
                val_preds = []
                valid_loss = 0
                for x, y in valid_loader:
                    output = model(x)  # Maintain hidden state across time steps
                    y = y.squeeze().float()
                    loss = criterion(output, y)
                    valid_loss += loss.item()
                    val_preds.append(output.detach().numpy())

                # Compute metrics
                valid_loss /= len(valid_loader)
                val_preds = np.concatenate(val_preds, axis=0)  # Ensure val_preds is a flat array
                val_preds = np.array(val_preds)
                valid_aucroc = roc_auc_score(y_val, val_preds)
                valid_aucpr = average_precision_score(y_val, val_preds)
                val_pred_labels = [1 if prob >= 0.5 else 0 for prob in val_preds]  # Since accuracy requires labels
                valid_accuracy = accuracy_score(y_val, val_pred_labels)
                
            # Log each epoch results
            wandb.log({
                'epoch': epoch,
                'train_loss': train_loss,
                'valid_loss': valid_loss,
                'train_aucroc': train_aucroc,
                'valid_aucroc': valid_aucroc,
                'train_aucpr': train_aucpr,
                'valid_aucpr': valid_aucpr,
                'train_accuracy': train_accuracy,
                'valid_accuracy': valid_accuracy
            })

            # Optimize for the best aucroc
            if valid_aucroc >= curr_batch_best_aucroc:
                best_model = model
                # Save for dataframe
                best_moment_row = {
                    'run_id': run.name,  # For checking Wandb Logs
                    'activation': activation,
                    'seed': seed,
                    'hidden_dim_1': hidden_1,
                    'hidden_dim_2': hidden_2,
                    'mlp_dim': mlp_dim,
                    'learning_rate': lr_val,
                    'dropout': dropout,
                    'l2_regularization': l2_val,
                    'num_layers_LSTM': num_layer_LSTM,
                    'num_layers_GRU': num_layer_GRU,
                    'trained_epochs': epoch + 1,
                    'train_loss': train_loss,
                    'valid_loss': valid_loss,
                    'train_aucroc': train_aucroc,
                    'valid_aucroc': valid_aucroc,
                    'train_aucpr': train_aucpr,
                    'valid_aucpr': valid_aucpr,
                    'train_accuracy': train_accuracy,
                    'valid_accuracy': valid_accuracy
                }
                
                # Save the model
                curr_batch_best_aucroc = valid_aucroc
                
            
            # Early stopping only after 50 epochs
            if epoch >= 50:
                if valid_aucroc >= curr_batch_best_aucroc:
                    no_improvement_counter = 0
                    curr_batch_best_aucroc = valid_aucroc
                else:
                    no_improvement_counter += 1
                    
                if no_improvement_counter == patience:
                    print(f'Training ending at epoch number: {epoch + 1}')
                    break
                
            # Display current results
            if epoch % 5 - 4 == 0:
                print(f"""
                    Epoch {epoch+1}/{num_epochs}:\n\tTrain Loss: {train_loss}, Validation Loss: {valid_loss}\n\tTrain AUCROC: {train_aucroc}, Validation AUCROC: {valid_aucroc}\n\tTrain AUCPR: {train_aucpr}, Validation AUCPR: {valid_aucpr}\n\tTrain Accuracy: {train_accuracy}, Validation Accuracy: {valid_accuracy}\n
                """)

        test_loss, test_aucroc, test_aucpr, test_accuracy = best_model.test_model_binary(best_model, test_loader, criterion, y_test)

        best_moment_row['test_loss'] = test_loss
        best_moment_row['test_aucroc'] = test_aucroc
        best_moment_row['test_aucpr'] = test_aucpr
        best_moment_row['test_accuracy'] = test_accuracy

        # Save the best moment from this training
        pd.DataFrame([best_moment_row]).to_csv(csv_file_path, mode='a', header=False, index=False)
        
wandb.finish()

Data Checking

In [7]:
import pandas as pd
import numpy as np


pd.set_option('display.max_columns', None)  # Ensure all columns are displayed
pd.set_option('display.width', 1000)  # Ensure rows are displayed in a single line

In [8]:
def compute_score(group_df):
    scores_optim = []
    train_aucs = []
    val_aucs = []
    test_aucs = []
    
    for idx, row in group_df.iterrows():
        optim_score = (row['train_aucroc'] * 0.3) + (row['valid_aucroc'] * 0.7)
        scores_optim.append(optim_score)
        train_aucs.append(row['train_aucroc'])
        val_aucs.append(row['valid_aucroc'])
        test_aucs.append(row['test_aucroc'])
         
    return sum(scores_optim) / len(scores_optim), np.mean(train_aucs), np.mean(val_aucs), np.mean(test_aucs)
    
    
df = pd.read_csv('data/output/results/BinaryTesting/data/probabilities_gs/probabilites_gs_multidata.csv')

df['grouping_key'] = df['run_id'].apply(lambda x: x.split('_')[-1])
grouped_df = df.groupby('grouping_key')
group_scores = []
for group_key, group_df in grouped_df:
    score, train_auc, val_auc, test_auc = compute_score(group_df)
    group_scores.append({'group': group_key, 'score': score, 'avg_train_auc': train_auc, 'avg_val_auc': val_auc, 'avg_test_auc': test_auc})


scores_df = pd.DataFrame(group_scores)

# Sort scores from highest to lowest
sorted_scores = scores_df.sort_values(by='score', ascending=False)

print(sorted_scores.head(50))

    group     score  avg_train_auc  avg_val_auc  avg_test_auc
307   375  0.853034       0.721275     0.909502      0.499265
413   470  0.852616       0.766292     0.889613      0.544755
136   220  0.850070       0.717186     0.907020      0.481893
63    155  0.849698       0.778940     0.880023      0.546432
108   196  0.847206       0.737156     0.894370      0.592472
222   299  0.846969       0.754614     0.886550      0.500673
521    74  0.846926       0.748602     0.889065      0.522795
121   207  0.846924       0.735943     0.894487      0.421467
288   358  0.846755       0.763463     0.882452      0.529233
348   411  0.846744       0.759790     0.884010      0.452183
250   323  0.845688       0.743682     0.889405      0.575432
151   234  0.844468       0.669345     0.919521      0.538629
117   203  0.844340       0.715428     0.899589      0.519539
319   386  0.843900       0.701135     0.905085      0.486271
6     103  0.843271       0.759429     0.879203      0.530090
138   22

In [9]:
# Sort scores from highest to lowest
sorted_scores = scores_df.sort_values(by='avg_val_auc', ascending=False)

print(sorted_scores.head(50))

    group     score  avg_train_auc  avg_val_auc  avg_test_auc
151   234  0.844468       0.669345     0.919521      0.538629
307   375  0.853034       0.721275     0.909502      0.499265
356   419  0.827118       0.635989     0.909030      0.533371
137   221  0.841195       0.683921     0.908598      0.533229
462   514  0.836780       0.670094     0.908217      0.599036
458   510  0.842402       0.690819     0.907366      0.641415
87    177  0.811671       0.589077     0.907068      0.589832
136   220  0.850070       0.717186     0.907020      0.481893
319   386  0.843900       0.701135     0.905085      0.486271
546    97  0.835919       0.674968     0.904898      0.545323
244   318  0.811937       0.595867     0.904538      0.441809
171   252  0.834571       0.674458     0.903191      0.530824
186   266  0.800853       0.563701     0.902490      0.655096
138   222  0.843261       0.705594     0.902261      0.535137
437   492  0.827297       0.656335     0.900566      0.457729
254   32

In [19]:
df = pd.read_csv('data/output/results/BinaryTesting/data/embedding_testing_parallel_new2.csv')

bc_df = df[df['activation'] == 'Betweenness_Closeness']
b_df = df[df['activation'] == 'Betweenness']
c_df = df[df['activation'] == 'Closeness']

bc_df = bc_df[bc_df['valid_aucroc'] >= 0.85]
b_df = b_df[b_df['valid_aucroc'] >= 0.85]
c_df = c_df[c_df['valid_aucroc'] >= 0.85]

bc_tmp_df_1 = bc_df[bc_df['batch_size'] == 1]
bc_tmp_df_16 = bc_df[bc_df['batch_size'] == 16]
bc_tmp_df_32 = bc_df[bc_df['batch_size'] == 32]
bc_tmp_df_64 = bc_df[bc_df['batch_size'] == 64]

b_tmp_df_1 = b_df[b_df['batch_size'] == 1]
b_tmp_df_16 = b_df[b_df['batch_size'] == 16]
b_tmp_df_32 = b_df[b_df['batch_size'] == 32]
b_tmp_df_64 = b_df[b_df['batch_size'] == 64]

c_tmp_df_1 = c_df[c_df['batch_size'] == 1]
c_tmp_df_16 = c_df[c_df['batch_size'] == 16]
c_tmp_df_32 = c_df[c_df['batch_size'] == 32]
c_tmp_df_64 = c_df[c_df['batch_size'] == 64]

print('BC train')
print(bc_tmp_df_1['train_aucroc'].mean())
print(bc_tmp_df_16['train_aucroc'].mean())
print(bc_tmp_df_32['train_aucroc'].mean())
print(bc_tmp_df_64['train_aucroc'].mean())
print()
print('B train')
print(b_tmp_df_1['train_aucroc'].mean())
print(b_tmp_df_16['train_aucroc'].mean())
print(b_tmp_df_32['train_aucroc'].mean())
print(b_tmp_df_64['train_aucroc'].mean())
print()
print('C train')
print(c_tmp_df_1['train_aucroc'].mean())
print(c_tmp_df_16['train_aucroc'].mean())
print(c_tmp_df_32['train_aucroc'].mean())
print(c_tmp_df_64['train_aucroc'].mean())
print()
print()

print('BC val')
print(bc_tmp_df_1['valid_aucroc'].mean())
print(bc_tmp_df_16['valid_aucroc'].mean())
print(bc_tmp_df_32['valid_aucroc'].mean())
print(bc_tmp_df_64['valid_aucroc'].mean())
print()
print('B val')
print(b_tmp_df_1['valid_aucroc'].mean())
print(b_tmp_df_16['valid_aucroc'].mean())
print(b_tmp_df_32['valid_aucroc'].mean())
print(b_tmp_df_64['valid_aucroc'].mean())
print()
print('C val')
print(c_tmp_df_1['valid_aucroc'].mean())
print(c_tmp_df_16['valid_aucroc'].mean())
print(c_tmp_df_32['valid_aucroc'].mean())
print(c_tmp_df_64['valid_aucroc'].mean())
print()
print()

print('BC Test')
print(bc_tmp_df_1['test_aucroc'].mean())
print(bc_tmp_df_16['test_aucroc'].mean())
print(bc_tmp_df_32['test_aucroc'].mean())
print(bc_tmp_df_64['test_aucroc'].mean())
print()
print('B Test')
print(b_tmp_df_1['test_aucroc'].mean())
print(b_tmp_df_16['test_aucroc'].mean())
print(b_tmp_df_32['test_aucroc'].mean())
print(b_tmp_df_64['test_aucroc'].mean())
print()
print('C Test')
print(c_tmp_df_1['test_aucroc'].mean())
print(c_tmp_df_16['test_aucroc'].mean())
print(c_tmp_df_32['test_aucroc'].mean())
print(c_tmp_df_64['test_aucroc'].mean())
print()
print()

BC train
0.40585189147136047
0.9010381211708645
0.9387125595643293
0.8230088495575221

B train
0.47119639210347175
nan
0.6982641252552758
nan

C train
0.39203965282505104
0.982215793056501
0.92357045609258
nan


BC val
0.9081890331890332
0.981060606060606
0.9232954545454545
0.8920454545454546

B val
0.9255050505050505
nan
0.8535353535353536
nan

C val
0.9482323232323233
0.9015151515151516
0.9242424242424244
nan


BC Test
0.6568027210884354
0.2392857142857142
0.6035714285714286
0.680952380952381

B Test
0.4660714285714285
nan
0.4666666666666667
nan

C Test
0.6089285714285714
0.3904761904761904
0.5785714285714286
nan




In [ ]:
df = pd.read_csv('data/output/results/BinaryTesting/data/embedding_testing_parallel.csv')


In [2]:
import pandas as pd
df = pd.read_csv('data/output/results/BinaryTesting/data/embedding_testing_parallel_new.csv')

tmp_df = df.nlargest(50, 'test_aucroc')  # Get the rows with the highest n values
print(tmp_df)
'''
bc_df = df[df['activation'] == 'Betweenness_Closeness']
b_df = df[df['activation'] == 'Betweenness']
c_df = df[df['activation'] == 'Closeness']

bc_df = bc_df[bc_df['valid_aucroc'] >= 0.85]
b_df = b_df[b_df['valid_aucroc'] >= 0.85]
c_df = c_df[c_df['valid_aucroc'] >= 0.85]


print('BC train')
print(bc_df['train_aucroc'].mean())
print('BC val')
print(bc_df['valid_aucroc'].mean())
print('BC Test')
print(bc_df['test_aucroc'].mean())

print()
print('B train')
print(b_df['train_aucroc'].mean())
print('B val')
print(b_df['valid_aucroc'].mean())
print('B Test')
print(b_df['test_aucroc'].mean())
print()

print('C train')
print(c_df['train_aucroc'].mean())
print('C val')
print(c_df['valid_aucroc'].mean())
print('C Test')
print(c_df['test_aucroc'].mean())
print()'''

                                     run_id        dataset             activation  seed  normalization  hidden_size_rnn  hidden_size_other  learning_rate  dropout  l2_regularization  num_layers                                   combo  trained_epochs  train_loss  valid_loss  test_loss  train_aucroc  valid_aucroc  test_aucroc  train_aucpr  valid_aucpr  test_aucpr  train_accuracy  valid_accuracy  test_accuracy
9              networkbancor_Betweenness_28  networkbancor            Betweenness    42          False             1000               1000         0.0010        0            0.00001           3   ['GRU', 'Attention', 'FC', 'Sigmoid']              57    0.723819    0.775955   0.472260      1.000000      0.481902     1.000000     0.456221     0.234043    0.637796        0.266667        0.646935       0.744681
50             networkbancor_Betweenness_76  networkbancor            Betweenness    42          False             1000               2500         0.0010        0            0.00

"\nbc_df = df[df['activation'] == 'Betweenness_Closeness']\nb_df = df[df['activation'] == 'Betweenness']\nc_df = df[df['activation'] == 'Closeness']\n\nbc_df = bc_df[bc_df['valid_aucroc'] >= 0.85]\nb_df = b_df[b_df['valid_aucroc'] >= 0.85]\nc_df = c_df[c_df['valid_aucroc'] >= 0.85]\n\n\nprint('BC train')\nprint(bc_df['train_aucroc'].mean())\nprint('BC val')\nprint(bc_df['valid_aucroc'].mean())\nprint('BC Test')\nprint(bc_df['test_aucroc'].mean())\n\nprint()\nprint('B train')\nprint(b_df['train_aucroc'].mean())\nprint('B val')\nprint(b_df['valid_aucroc'].mean())\nprint('B Test')\nprint(b_df['test_aucroc'].mean())\nprint()\n\nprint('C train')\nprint(c_df['train_aucroc'].mean())\nprint('C val')\nprint(c_df['valid_aucroc'].mean())\nprint('C Test')\nprint(c_df['test_aucroc'].mean())\nprint()"

In [4]:
tmp_df_test = df.nlargest(50, 'test_aucroc')  # Get the rows with the highest n values
print(tmp_df_test)

                                     run_id        dataset             activation  seed  normalization  hidden_size_rnn  hidden_size_other  learning_rate  dropout  l2_regularization  num_layers                                   combo  trained_epochs  train_loss  valid_loss  test_loss  train_aucroc  valid_aucroc  test_aucroc  train_aucpr  valid_aucpr  test_aucpr  train_accuracy  valid_accuracy  test_accuracy
9              networkbancor_Betweenness_28  networkbancor            Betweenness    42          False             1000               1000         0.0010        0            0.00001           3   ['GRU', 'Attention', 'FC', 'Sigmoid']              57    0.723819    0.775955   0.472260      1.000000      0.481902     1.000000     0.456221     0.234043    0.637796        0.266667        0.646935       0.744681
50             networkbancor_Betweenness_76  networkbancor            Betweenness    42          False             1000               2500         0.0010        0            0.00

In [14]:
tmp_df_train = df.nlargest(num_rows, 'train_aucroc')  # Get the rows with the highest n values
print(tmp_df_train)

                                     run_id        dataset             activation  seed  normalization  hidden_size_rnn  hidden_size_other  learning_rate  dropout  l2_regularization  num_layers                                   combo  trained_epochs  train_loss  valid_loss  test_loss  train_aucroc  valid_aucroc  test_aucroc  train_aucpr  valid_aucpr  test_aucpr  train_accuracy  valid_accuracy  test_accuracy
9              networkbancor_Betweenness_28  networkbancor            Betweenness    42          False             1000               1000         0.0010        0            0.00001           3   ['GRU', 'Attention', 'FC', 'Sigmoid']              57    0.723819    0.775955   0.472260      1.000000      0.481902     1.000000     0.456221     0.234043    0.637796        0.266667        0.646935       0.744681
50             networkbancor_Betweenness_76  networkbancor            Betweenness    42          False             1000               2500         0.0010        0            0.00

In [18]:
#tmp_df = df[df['valid_aucroc'] >= 0.80]

df = pd.read_csv('data/output/results/BinaryTesting/data/embedding_testing_parallel_new.csv')
tmp_df = df[df['valid_aucroc'] >= 0.750]
params = ['activation', 'seed', 'normalization', 'hidden_size_rnn', 'hidden_size_other', 'learning_rate', 'dropout', 'l2_regularization', 'num_layers', 'combo']

print('The following results are for all models with a valid_aucroc >= 0.90')
print(f'The average train_aucroc is: {tmp_df['train_aucroc'].mean()}')
print(f'The average test_aucroc is: {tmp_df['test_aucroc'].mean()}')

for param in params:
    metric = tmp_df[param].value_counts()
    print(f'For param: {param}, the value_counts() is: {metric}')
    

The following results are for all models with a valid_aucroc >= 0.90
The average train_aucroc is: 0.7478723890488598
The average test_aucroc is: 0.5357326117805177
For param: activation, the value_counts() is: activation
Betweenness_Closeness    51
Betweenness              34
Closeness                32
Name: count, dtype: int64
For param: seed, the value_counts() is: seed
42    117
Name: count, dtype: int64
For param: normalization, the value_counts() is: normalization
False    117
Name: count, dtype: int64
For param: hidden_size_rnn, the value_counts() is: hidden_size_rnn
1000    117
Name: count, dtype: int64
For param: hidden_size_other, the value_counts() is: hidden_size_other
1000    48
2500    42
4000    19
8000     8
Name: count, dtype: int64
For param: learning_rate, the value_counts() is: learning_rate
0.0001    115
0.0010      2
Name: count, dtype: int64
For param: dropout, the value_counts() is: dropout
0    117
Name: count, dtype: int64
For param: l2_regularization, the val

In [ ]:
model_combos = ["['LSTM', 'MLP', 'Sigmoid']", "['GRU', 'MLP', 'Sigmoid']", "['LSTM', 'FC', 'Sigmoid']", "['GRU', 'FC', 'Sigmoid']", 
                    "['GRU', 'Attention', 'FC', 'Sigmoid']", "['LSTM', 'Attention', 'FC', 'Sigmoid']", "['LSTM', 'GRU', 'FC', 'Sigmoid']", "['LSTM', 'GRU', 'MLP', 'Sigmoid']"]
activations = ['Degree_Forman_Weight', 'Forman', 'Forman_Weight', 'Degree',  'Degree_Forman', 'Weight', 'Degree_Weight']
scores = ['valid_aucroc', 'test_aucroc', 'valid_aucpr', 'test_aucpr', 'valid_accuracy', 'test_accuracy']

for combo in model_combos:
    for metric in scores:
        tmp_df = df[df['combo'] == combo]
        score = tmp_df[metric].mean()
        print(f'{combo} has average {metric} of: {score}')
    print()  # Formatting
        
for activation in activations:
    for metric in scores:
        tmp_df = df[df['activation'] == activation]
        score = tmp_df[metric].mean()
        print(f'{activation} has average {metric} of: {score}')
    print()  # Formatting

In [ ]:
df = pd.read_csv('data/output/results/BinaryTesting/data/embedding_testing_best_parallel.csv')
# Model 1
tmp_df = df[df['seed'] == 42]
tmp_df = tmp_df[tmp_df['activation'] == 'Forman_Weight']
tmp_df = tmp_df[tmp_df['normalization'] == False]
tmp_df = tmp_df[tmp_df['combo'] == "['GRU', 'Attention', 'FC', 'Sigmoid']"]
tmp_df = tmp_df[tmp_df['l2_regularization'] == 0.0001]
tmp_df = tmp_df[tmp_df['learning_rate'] == 0.0001]
tmp_df = tmp_df[tmp_df['dropout'] == 0.35]
tmp_df = tmp_df[tmp_df['hidden_size_rnn'] == 128]
tmp_df = tmp_df[tmp_df['hidden_size_other'] == 32]
print(tmp_df)
print()

In [10]:
from utils.loader import Loader
from utils.utils import Utils

my_loader = Loader()
my_utils = Utils()

datasets = ['networkadex', 'networkbancor', 'networkcentra', 'networkcoindash',  
                'mathoverflow', 'networkaeternity', 'Reddit_B',  'networkaragon', 'networkcindicator', 'networkiconomi', 'CollegeMsg', 'networkdgd', 'networkaion']

for dataset in datasets:
    print(f'Using dataset {dataset}')
    
    embeddings, labels = my_loader.load_data(dataset, 'Degree')  # Load embeddings and labels
                
    n = len(embeddings)

    # Calculate split indices
    train_end = int(0.7 * n)  # 70% for training
    val_end = int(0.85 * n)   # Next 15% for validation (70% + 15% = 85%)
    X_train, y_train = embeddings[:train_end], labels[:train_end]
    X_val, y_val = embeddings[train_end:val_end], labels[train_end:val_end]
    X_test, y_test = embeddings[val_end:], labels[val_end:]
    
    print(f'There are {y_train.count(0)} 0\'s and {y_train.count(1)} 1\'s in training')
    print(f'There are {y_val.count(0)} 0\'s and {y_val.count(1)} 1\'s in val')
    print(f'There are {y_test.count(0)} 0\'s and {y_test.count(1)} 1\'s in test')

Using dataset networkadex
There are 116 0's and 89 1's in training
There are 34 0's and 10 1's in val
There are 20 0's and 24 1's in test
Using dataset networkbancor
There are 104 0's and 113 1's in training
There are 36 0's and 11 1's in val
There are 12 0's and 35 1's in test
Using dataset networkcentra
There are 87 0's and 95 1's in training
There are 29 0's and 10 1's in val
There are 20 0's and 20 1's in test
Using dataset networkcoindash
There are 106 0's and 81 1's in training
There are 26 0's and 14 1's in val
There are 14 0's and 27 1's in test
Using dataset mathoverflow
There are 74 0's and 54 1's in training
There are 11 0's and 16 1's in val
There are 10 0's and 18 1's in test
Using dataset networkaeternity
There are 88 0's and 72 1's in training
There are 24 0's and 10 1's in val
There are 11 0's and 24 1's in test
Using dataset Reddit_B
There are 141 0's and 138 1's in training
There are 17 0's and 43 1's in val
There are 31 0's and 29 1's in test
Using dataset networkara

In [5]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

from utils.embedding_methods.betweenness import EmbedBetweenness
from utils.embedding_methods.closeness import EmbedCloseness
from utils.embedding_methods.degree import EmbedDegree
from utils.embedding_methods.forman_ricci import EmbedForman
from utils.embedding_methods.weight import EmbedWeight

activations_list = [[EmbedBetweenness, EmbedCloseness], [EmbedBetweenness], [EmbedCloseness],  [EmbedDegree, EmbedBetweenness], [EmbedForman, EmbedBetweenness], [EmbedDegree, EmbedCloseness], [EmbedForman, EmbedCloseness], [EmbedDegree, EmbedForman, EmbedWeight],[EmbedForman], [EmbedForman, EmbedWeight], [EmbedDegree],  [EmbedDegree, EmbedForman], [EmbedWeight], [EmbedDegree, EmbedWeight], [EmbedWeight], ]
activation_combos_names = ['Betweenness_Closeness', 'Betweenness', 'Closeness',  'Degree_Betweenness', 'Forman_Betweenness', 'Degree_Closeness', 'Forman_Closeness', 'Degree_Forman_Weight', 'Forman', 'Forman_Weight', 'Degree',  'Degree_Forman', 'Weight', 'Degree_Weight' ]
datasets = ['networkbancor', 'networkadex', 'networkcentra', 'networkcoindash',  'mathoverflow', 'networkaeternity', 'Reddit_B',  'networkaragon', ]

my_utils = Utils()
my_loader = Loader()

for dataset in datasets:
    for activations in activations_list:
        input_dim = 0
        # Set up embeddings
        embeddings = None  # Init
        for activation in activations:
            tmp_activation_name = my_utils.get_activation_name(activation)
            data, labels = my_loader.load_data(dataset, tmp_activation_name)  # Load embeddings and labels
            embeddings = my_utils.concat_embeddings(embeddings, data)  # Add the new data
            
            input_dim += 30  # To account for changing embeddings
            
        # Split data 70/15/15
        n = len(embeddings)

        # Calculate split indices
        train_end = int(0.7 * n)  # 70% for training
        val_end = int(0.85 * n)   # Next 15% for validation (70% + 15% = 85%)
        X_train, y_train = embeddings[:train_end], labels[:train_end]
        X_val, y_val = embeddings[train_end:val_end], labels[train_end:val_end]
        X_test, y_test = embeddings[val_end:], labels[val_end:]

        param_grid = {
            'n_estimators': [50, 100, 150, 200],  # Number of trees in the forest
            'max_depth': [None, 10, 20, 30],  # Maximum depth of the tree
            'min_samples_split': [2, 5, 10],  # Minimum samples required to split an internal node
            'min_samples_leaf': [1, 2, 4],  # Minimum number of samples required to be at a leaf node
            'bootstrap': [True, False]  # Whether bootstrap samples are used when building trees
        }
        # Set up GridSearchCV with RandomForestClassifier and the parameter grid
        rf = RandomForestClassifier(random_state=42)
        grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, scoring='roc_auc', n_jobs=-1, verbose=10)

        # Perform the grid search
        grid_search.fit(X_train, y_train)

        # Get the best parameters and the corresponding score
        print(f"Working on {dataset} with activation {activations}")
        print(f"Best Hyperparameters: {grid_search.best_params_}")
        print(f"Best AUCROC Score: {grid_search.best_score_}")

        # Use the best model found by GridSearchCV to predict on the test set
        best_model = grid_search.best_estimator_
        y_pred_prob = best_model.predict_proba(X_test)[:, 1]  # Get probabilities for ROC AUC
        test_aucroc = roc_auc_score(y_test, y_pred_prob)

        print(f"Test AUCROC Score: {test_aucroc}")

Fitting 5 folds for each of 288 candidates, totalling 1440 fits


c:\Users\ronan\anaconda3\envs\temporal_env\Lib\site-packages\numpy\ma\core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


Working on networkbancor with activation [<class 'utils.embedding_methods.betweenness.EmbedBetweenness'>, <class 'utils.embedding_methods.closeness.EmbedCloseness'>]
Best Hyperparameters: {'bootstrap': True, 'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 150}
Best AUCROC Score: 0.749896480331263
Test AUCROC Score: 0.34523809523809523
Fitting 5 folds for each of 288 candidates, totalling 1440 fits
Working on networkbancor with activation [<class 'utils.embedding_methods.betweenness.EmbedBetweenness'>]
Best Hyperparameters: {'bootstrap': True, 'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 200}
Best AUCROC Score: 0.8128910220214568
Test AUCROC Score: 0.3321428571428572
Fitting 5 folds for each of 288 candidates, totalling 1440 fits
Working on networkbancor with activation [<class 'utils.embedding_methods.closeness.EmbedCloseness'>]
Best Hyperparameters: {'bootstrap': True, 'max_depth': 10, 'min_samples_leaf': 2, 'min_s

c:\Users\ronan\anaconda3\envs\temporal_env\Lib\site-packages\numpy\ma\core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


Working on networkbancor with activation [<class 'utils.embedding_methods.degree.EmbedDegree'>, <class 'utils.embedding_methods.betweenness.EmbedBetweenness'>]
Best Hyperparameters: {'bootstrap': False, 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 100}
Best AUCROC Score: 0.841089779785432
Test AUCROC Score: 0.3119047619047619
Fitting 5 folds for each of 288 candidates, totalling 1440 fits
Working on networkbancor with activation [<class 'utils.embedding_methods.forman_ricci.EmbedForman'>, <class 'utils.embedding_methods.betweenness.EmbedBetweenness'>]
Best Hyperparameters: {'bootstrap': True, 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 50}
Best AUCROC Score: 0.7979023150762281
Test AUCROC Score: 0.40595238095238095
Fitting 5 folds for each of 288 candidates, totalling 1440 fits
Working on networkbancor with activation [<class 'utils.embedding_methods.degree.EmbedDegree'>, <class 'utils.embedding_methods.closene

...

Working on mathoverflow with activation [<class 'utils.embedding_methods.degree.EmbedDegree'>, <class 'utils.embedding_methods.forman_ricci.EmbedForman'>]
Best Hyperparameters: {'bootstrap': True, 'max_depth': None, 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 50}
Best AUCROC Score: 0.511099567099567
Test AUCROC Score: 0.85
Fitting 5 folds for each of 288 candidates, totalling 1440 fits


c:\Users\ronan\anaconda3\envs\temporal_env\Lib\site-packages\numpy\ma\core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


Working on mathoverflow with activation [<class 'utils.embedding_methods.weight.EmbedWeight'>]
Best Hyperparameters: {'bootstrap': True, 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 200}
Best AUCROC Score: 0.5463376623376623
Test AUCROC Score: 0.26666666666666666
Fitting 5 folds for each of 288 candidates, totalling 1440 fits
Working on mathoverflow with activation [<class 'utils.embedding_methods.degree.EmbedDegree'>, <class 'utils.embedding_methods.weight.EmbedWeight'>]
Best Hyperparameters: {'bootstrap': False, 'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 50}
Best AUCROC Score: 0.5315064935064935
Test AUCROC Score: 0.22222222222222224
Fitting 5 folds for each of 288 candidates, totalling 1440 fits
Working on mathoverflow with activation [<class 'utils.embedding_methods.weight.EmbedWeight'>]
Best Hyperparameters: {'bootstrap': True, 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators

c:\Users\ronan\anaconda3\envs\temporal_env\Lib\site-packages\sklearn\model_selection\_validation.py:540: FitFailedWarning: 
1152 fits failed out of a total of 1440.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1152 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\ronan\anaconda3\envs\temporal_env\Lib\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\ronan\anaconda3\envs\temporal_env\Lib\site-packages\sklearn\base.py", line 1473, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\ronan\anaconda3\envs\temporal_env\Lib\site

ValueError: Input X contains infinity or a value too large for dtype('float32').